In [1]:
import sys
from pathlib import Path
import pandas as pd

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [2]:
# Extract canonical facts from FCA HTML files
from src.parsing.arelle_parser import process_html_files

html_folder = projectRoot / "data" / "raw" / "fca"
output_path = (projectRoot / "data" / "processed" / 
               "canonical_facts" / "bronze.csv")

if output_path.exists():
    print(f"Bronze data already exists at: {output_path}")
    bronze_df = pd.read_csv(output_path)
else:
    bronze_df = process_html_files(html_folder, output_path, debug=False)

sample_bronze = bronze_df.sample(n=250, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_bronze.csv")
sample_bronze.to_csv(csv_path, index=False)
print(f"Bronze sample extract saved to: {csv_path}")

Bronze data already exists at: /workspace/data/processed/canonical_facts/bronze.csv
Bronze sample extract saved to: /workspace/data/processed/debug/sample_bronze.csv


In [3]:
# Create silver ground truth from bronze data
from src.parsing.canonical_facts import create_silver_ground_truth

silver_df = create_silver_ground_truth(bronze_df, 
                        str(projectRoot / "data" / "processed" /
                            "canonical_facts" / "silver.csv"))

sample_silver = silver_df.sample(n=300, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_silver.csv")
sample_silver.to_csv(csv_path, index=False)
print(f"Silver sample extract saved to: {csv_path}")

Created cleaned ground truth with 5174 rows.
Silver data saved to: /workspace/data/processed/canonical_facts/silver.csv
Silver sample extract saved to: /workspace/data/processed/debug/sample_silver.csv


In [4]:
# Create gold ground truth from silver data
from src.parsing.canonical_facts import create_gold_ground_truth

gold_df = create_gold_ground_truth(silver_df, 
                         str(projectRoot / "data" / "processed" /
                              "canonical_facts" / "gold.csv"))

sample_gold = gold_df.sample(n=300, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_gold.csv")
sample_gold.to_csv(csv_path, index=False)
print(f"Gold sample extract saved to: {csv_path}")

Gold data saved to: /workspace/data/processed/canonical_facts/gold.csv
Gold sample extract saved to: /workspace/data/processed/debug/sample_gold.csv


In [5]:
# Generate TINY gold sample
sample_gold_tiny = gold_df[~gold_df['segment'].isin(['Narrative_Disclosure', 
                        'Other_Financial_Metric'])].sample(n=10, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_gold_tiny.csv")
sample_gold_tiny.to_csv(csv_path, index=False)
print(f"Tiny Gold sample extract saved to: {csv_path}")

Tiny Gold sample extract saved to: /workspace/data/processed/debug/sample_gold_tiny.csv


In [6]:
from src.qa_generation.llm_qa_generator import generate_qa_openai
input_csv_path = (projectRoot / "data" / "processed" /
                  "debug" / "sample_gold_tiny.csv")
output_csv_path = (projectRoot / "data" / "qa" /
                   "debug" / "sample_qa_pairs_tiny.csv")

if output_csv_path.exists():
    print(f"QA pairs already exist at: {output_csv_path}")
    qa_pairs_df = pd.read_csv(output_csv_path)
else:
    qa_pairs_df = generate_qa_openai(input_csv_path, output_csv_path)

QA pairs already exist at: /workspace/data/qa/debug/sample_qa_pairs_tiny.csv


In [10]:
# Evaluate models on QA pairs
input_qa_pairs_path = (projectRoot / "data" / "qa" /
                        "debug" / "sample_qa_pairs_tiny.csv")
evaluation_output_path = (projectRoot / "data" / "results" /
                          "debug" / "sample_qa_evaluation_tiny.csv")
from src.evaluation.llm_interface import evaluate_models_on_qa_pairs

evaluation_df = evaluate_models_on_qa_pairs(input_qa_pairs_path, 
                                      evaluation_output_path)

Loaded 10 QA pairs
✓ gpt-4o-mini - Row 0
✓ llama-3.3-70b-versatile - Row 0
✗ openai/gpt-oss-120b - Row 0: Expecting value: line 1 column 1 (char 0)
✓ gpt-4o-mini - Row 1
✓ llama-3.3-70b-versatile - Row 1
✗ openai/gpt-oss-120b - Row 1: Expecting value: line 1 column 1 (char 0)
✓ gpt-4o-mini - Row 2
✓ llama-3.3-70b-versatile - Row 2
✗ openai/gpt-oss-120b - Row 2: Expecting value: line 1 column 1 (char 0)
✓ gpt-4o-mini - Row 3
✓ llama-3.3-70b-versatile - Row 3
✗ openai/gpt-oss-120b - Row 3: Expecting value: line 1 column 1 (char 0)
✓ gpt-4o-mini - Row 4
✓ llama-3.3-70b-versatile - Row 4
✗ openai/gpt-oss-120b - Row 4: Expecting value: line 1 column 1 (char 0)

--- Evaluated 5/10 QA pairs ---

✓ gpt-4o-mini - Row 5
✓ llama-3.3-70b-versatile - Row 5
✗ openai/gpt-oss-120b - Row 5: Expecting value: line 1 column 1 (char 0)
✓ gpt-4o-mini - Row 6
✓ llama-3.3-70b-versatile - Row 6
✗ openai/gpt-oss-120b - Row 6: Expecting value: line 1 column 1 (char 0)
✓ gpt-4o-mini - Row 7
✓ llama-3.3-70b-versat